In [2]:
import os 
import pandas as pd 
import numpy as np

from dotenv import load_dotenv

import requests 
import json 

from bs4 import BeautifulSoup
import io

import csv

In [ ]:
#### setting dictionaries - mappings 

In [24]:
import os
proj_root = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebooks/ -> project root
data_dir = os.path.join(proj_root, "data", "raw")
os.makedirs(data_dir, exist_ok=True)

In [26]:
import os, sys
print(data_dir)
print("cwd:", os.getcwd())
print("src exists:", os.path.exists(os.path.join(os.getcwd(), "src")))
print("sys.path[0]:", sys.path[0])

c:\Users\jachy\Desktop\Data-Processing-in-Python---Project\data\raw
cwd: c:\Users\jachy\Desktop\Data-Processing-in-Python---Project\notebooks
src exists: False
sys.path[0]: c:\Users\jachy\Desktop\Data-Processing-in-Python---Project


In [ ]:
### chmi weather stations - data processing

df_chmi_stat = pd.read_csv(os.path.join(data_dir, "chmi_stations_metadata.csv"))

df_chmi_stat = df_chmi_stat[df_chmi_stat['FULL_NAME'].str.contains('Praha', case=False, na=False)]

wsi_to_drop = [
    '0-203-0-11201020001', #Praha, Vinohrady - Flora	
    '0-203-0-11202007001', #Praha, Suchdol
    '0-203-0-11105048001', #Praha, Zadní Kopanina
    '0-203-0-11201020003' #Praha, Chodov
    ]

df_chmi_stat = df_chmi_stat[~df_chmi_stat['WSI'].isin(wsi_to_drop)]

df_chmi_stat["END_DATE_DT"] = pd.to_datetime(df_chmi_stat["END_DATE"], utc=True, errors="coerce")

df_chmi_stat = (
    df_chmi_stat.sort_values("END_DATE_DT")
    .drop_duplicates(subset="WSI", keep="last")
)

now_utc = pd.Timestamp.now(tz="UTC")
df_chmi_stat = df_chmi_stat[
    (df_chmi_stat["END_DATE_DT"] >= now_utc)
]

In [41]:
df_chmi_stat

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION,END_DATE_DT
62,0-20000-0-11518,P1PRUZ01,2000-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Ruzyně",14.255556,50.100278,364.00,3999-12-31 23:59:00+00:00
2214,0-203-0-11514,P1PKLE01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416923,50.086634,190.70,3999-12-31 23:59:00+00:00
1101,0-203-0-10904013001,P1PKOM01,2017-12-07T12:00:00Z,3999-12-31T23:59:00Z,"Praha, Komořany",14.406360,49.988600,213.00,3999-12-31 23:59:00+00:00
2217,0-203-0-11515,P1PKLM01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416436,50.086341,190.70,3999-12-31 23:59:00+00:00
68,0-20000-0-11520,P1PLIB01,2018-05-17T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Libuš",14.446944,50.007778,302.04,3999-12-31 23:59:00+00:00
1429,0-203-0-11201024001,P1PBRE01,2013-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Břevnov",14.352689,50.080843,355.00,3999-12-31 23:59:00+00:00
75,0-20000-0-11567,P1PKBE01,2010-10-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Kbely",14.538056,50.123333,284.50,3999-12-31 23:59:00+00:00
1425,0-203-0-11201020003,P1PCHO01,1997-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Chodov",14.500000,50.029400,297.00,3999-12-31 23:59:00+00:00
65,0-20000-0-11519,P1PKAR01,2024-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Karlov",14.427778,50.069167,260.50,3999-12-31 23:59:00+00:00
1273,0-203-0-11101007001,L4PRAH01,2015-03-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Brdy",13.818380,49.658890,862.00,3999-12-31 23:59:00+00:00


In [ ]:
### discionary of wsi codes

wsi_dict = dict(
    zip(
        df_chmi_stat["WSI"].astype(str),
        df_chmi_stat["FULL_NAME"].astype(str)  
    )
)

with open(os.path.join(data_dir, "wsi_dict.csv"),"w",encoding="utf-8-sig",newline="") as f:
    writer=csv.writer(f); writer.writerow(["key","value"]); writer.writerows(wsi_dict.items())

In [43]:
### chmi weather variables

### filtering only needed ones

df_chmi_vars = pd.read_csv(os.path.join(data_dir, "chmi_variables_metadata.csv"))

df_chmi_vars = df_chmi_vars[df_chmi_vars['WSI'].astype(str).isin(wsi_dict)]

### dictionary of variables abbreviations and names
chmi_vars_dict = dict(
    zip(
        df_chmi_vars['EG_EL_ABBREVIATION'].astype(str),
        df_chmi_vars['NAME'].astype(str)  
    )
)

with open(os.path.join(data_dir, "chmi_vars_dict.csv"),"w",encoding="utf-8-sig",newline="") as f:
    writer=csv.writer(f); writer.writerow(["key","value"]); writer.writerows(chmi_vars_dict.items())

In [ ]:
### chmi weather 10min data

df_weather = pd.read_csv(os.path.join(data_dir, "weather_data_10min.csv"))

# it gives flag warning, but i assume flag is not of interest


C:\Users\jachy\AppData\Local\Temp\ipykernel_26524\3649673168.py:3: DtypeWarning: Columns (0: FLAG) have mixed types. Specify dtype option on import or set low_memory=False.
  df_weather = pd.read_csv(os.path.join(data_dir, "weather_data_10min.csv"))


In [47]:
df_weather.head()

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH
0,0-20000-0-11518,Casmax,2025-01-01T00:00:00Z,273.0,NaN,0.0,0-20000-0-11518,2025,1
1,0-20000-0-11518,Casmax,2025-01-01T00:10:00Z,32.0,NaN,0.0,0-20000-0-11518,2025,1
2,0-20000-0-11518,Casmax,2025-01-01T00:20:00Z,1.0,NaN,0.0,0-20000-0-11518,2025,1
3,0-20000-0-11518,Casmax,2025-01-01T00:30:00Z,283.0,NaN,0.0,0-20000-0-11518,2025,1
4,0-20000-0-11518,Casmax,2025-01-01T00:40:00Z,1.0,NaN,0.0,0-20000-0-11518,2025,1


In [52]:
df_weather['FLAG'].isna().sum()

np.int64(5023823)

In [51]:
df_weather['FLAG'].notna().sum()

np.int64(21577)

In [ ]:
### mapping variables names in weather data 
df_weather['ELEMENT_NAME'] = df_weather['ELEMENT'].map(chmi_vars_dict)

In [ ]:
### golemio air quality 

#### air quality metadata processing

# station_cols = {
#     'geometry.coordinates': 'coordinates', 
#     'properties.id': 'id', 
#     'properties.name': 'name', 
#     'properties.district': 'district', 
#     'properties.measurement.components.type': 'components'
# }

# air_quality_stations = df[station_cols.keys()]

# air_quality_stations = air_quality_stations.rename(columns=station_cols)

# air_quality_stations = (
#     air_quality_stations.groupby('id', as_index=False)
#     .agg({
#         'coordinates': 'first',
#         'name': 'first',
#         'district': 'first',
#         'components': list
#     })
# )

# air_quality_stations[["lon", "lat"]] = pd.DataFrame(air_quality_stations["coordinates"].tolist(), index=air_quality_stations.index)
# air_quality_stations["lon"] = pd.to_numeric(air_quality_stations["lon"], errors="coerce")
# air_quality_stations["lat"] = pd.to_numeric(air_quality_stations["lat"], errors="coerce")


In [ ]:

#air quality stations dictionary

# air_stat_dict = dict(
#     zip(
#         air_quality_stations['id'].astype(str),
#         air_quality_stations['name'].astype(str)  
#     )
# )

In [ ]:
### loading the dictionaries


def load_wsi_dict(path="data/raw/wsi_dict.csv"):
    df = pd.read_csv(path, dtype=str)
    return dict(df.values)

def load_chmi_vars(path="data/raw/chmi_vars.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)